In [28]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive


In [29]:
# set env variables
TEST_DATASET = "SESAR_ZTC_test_multi_entire_filtered2.csv"
TRAIN_DATASET = "SESAR_ZTC_train_multi_entire_filtered2.csv" # used for example selection
OUTPUT_FILE = "OUTPUT.json"

# Set Up

In [30]:
!pip install datasets sentencepiece tokenizers bitsandbytes accelerate xformers einops
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-rig2kqmg
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-rig2kqmg
  Resolved https://github.com/huggingface/transformers to commit a25037beb9f039270b30a94c34ead72ea80ae8a5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Download

In [31]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

device = "cuda"
# This causes OOM

# model = AutoModelForCausalLM.from_pretrained(
#     "Open-Orca/Mistral-7B-OpenOrca").to(device)
# tokenizer = AutoTokenizer.from_pretrained(
#     "Open-Orca/Mistral-7B-OpenOrca")

In [32]:
import transformers
model_id = "Open-Orca/Mistral-7B-OpenOrca"
bnb_config = transformers.BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)

model = transformers.AutoModelForCausalLM.from_pretrained(
  model_id,
  trust_remote_code=True,
  quantization_config=bnb_config,
  device_map='auto',
)

tokenizer = transformers.AutoTokenizer.from_pretrained(
  model_id,
)
model.config.use_cache = True

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


# Inference

In [33]:
inputs = tokenizer(
    "Orcas were not known to be drawn to mistral energy, but they were seen recently ",
    return_tensors="pt").to(device)
outputs = model.generate(
    **inputs, max_new_tokens=256, use_cache=True, do_sample=True,
    temperature=0.2, top_p=0.95)
text = tokenizer.batch_decode(outputs)[0]
print(text)

Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.
Error during conversion: ValueError('Queue is full! Please try again.')


<s> Orcas were not known to be drawn to mistral energy, but they were seen recently  in the waters off the coast of France.

A pod of orcas, or killer whales, were seen swimming in the waters off the coast of France, seemingly attracted by the energy from a wind farm.

The unusual sighting was captured on video by a local resident, who noticed the orcas swimming in the area where the wind farm is located.

The wind farm, which is powered by the wind, generates electricity and is considered a form of renewable energy. It is believed that the orcas were attracted to the area due to the movement of the water caused by the wind turbines.

The orcas, which are known for their intelligence and curiosity, are often seen interacting with humans and other marine life. However, this is the first time they have been seen in such close proximity to a wind farm.

The video of the orcas swimming near the wind farm has sparked a debate among marine biologists and environmentalists about the potential

In [34]:
sys_prompt = "A chat."
prompt = "Tell me a joke."

prefix = "<|im_start|>"
suffix = "<|im_end|>\n"
sys_format = prefix + "system\n" + sys_prompt + suffix
user_format = prefix + "user\n" + prompt + suffix
assistant_format = prefix + "assistant\n"
input_text = sys_format + user_format + assistant_format

generation_config = GenerationConfig(
    max_length=256, temperature=1.1, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")

inputs = tokenizer(input_text, return_tensors="pt", return_attention_mask=True).to(device)
outputs = model.generate(**inputs, generation_config=generation_config)

text = tokenizer.batch_decode(outputs)[0]
print(text)

<s><|im_start|> system
A chat.<|im_end|><|im_start|> user
Tell me a joke.<|im_end|><|im_start|> assistant
 Why did the scarecrow win an award? 

Because the jury thought his speech was outstanding.<|im_end|>


In [35]:
print(text.split("<|im_end|>")[2][len("<|im_start|> assistant"):].strip("\n"))

 Why did the scarecrow win an award? 

Because the jury thought his speech was outstanding.


# Few Shot

In [36]:
# load test dataset
import pandas as pd
test_df = pd.read_csv(TEST_DATASET)

## Set Up - Read Taxonomy Files

In [37]:
import json
leaf_material_types = list(open("unique_leaf_labels.txt").read().splitlines())
joined_leaf_material_types = "\n".join(leaf_material_types)

leaf_to_entire_path_mapping = {}
with open('leaf_to_parents_mapping.json') as f:
    leaf_to_entire_path_mapping = json.load(f)

mapping = json.load(open("label_to_parent_mapping.json"))
label_to_parent = {}
for k, v in mapping.items():
  if "material" in v:
    v.remove("material")
  label_to_parent[k] = v

child_to_parent = json.load(open("child_to_parent_mapping.json"))
def get_parent_labels(curr_labels):
  # return the parent labels (upper level labels)
  parent_labels = []
  for label in curr_labels:
    if label in child_to_parent and child_to_parent[label]:
      parent_labels.append(child_to_parent[label])
  parent_labels = list(set(parent_labels))
  return parent_labels


In [38]:
import json

# mapping to description and individual fields that contain geology terms that need to be enriched
# this can be easily generated by using mapping of two columns in dataframe
desc_to_tax_map = json.load(open("description_to_taxonomy_train_all.json"))
desc_to_cm_map = json.load(open("description_to_collectionMethod_train_all.json"))
desc_to_desc_map = json.load(open("description_to_description_train_all.json"))

def trim_mapping(mapping):
  return {k.split("</s>")[0][len("<s>"):] : v for k,v in mapping.items()}
desc_to_tax_map = trim_mapping(desc_to_tax_map)
desc_to_cm_map = trim_mapping(desc_to_cm_map)
desc_to_desc_map = trim_mapping(desc_to_desc_map)

# Set Up - SentBERT

In [39]:
!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util
sentbert_model = SentenceTransformer("allenai-specter")

In [40]:
train_df = pd.read_csv(TRAIN_DATASET)
train_descriptions = list([x.split("</s>")[0][len("<s>"):] for x in train_df["concatenated_text_B"].tolist()])
descriptions_to_label = dict(zip(train_df['concatenated_text_B'], train_df['label_list']))
final_desc_to_label = {}
for k, v in descriptions_to_label.items():
  final_desc_to_label[k.split("</s>")[0][len("<s>"):]] = v
  final_desc_to_label[k.split("</s>")[0][len("<s>"):]+"."] = v
final_desc_to_label

{'The name of the specific place where the sample was collected is Windscoop Bluff, The object type of sample indicates that this sample is a sample that is an individual unit, including rock hand samples, a biological specimen, or a bottle of fluid, The age of a sample as described by the stratigraphic era, period, state, etc. is Neogene, The detailed description of the sample is Trachyte flow forming small bluff.  Cropping out SW of Camp III.  Finely vesicular, aphyric phonolite, The name of institution, museum, or repository where the sample is currently stored is Polar Rock Repository, Byrd Polar and Climate Research Center, Ohio State University, The taxonomy informal classification of sample is Phonolite, The method by which a sample was collected is Manual, The free text description of the related URL is None': 'sediment/igneous rock/natural solid material/phonolitoid/fine grained igneous rock/rock',
 'The name of the specific place where the sample was collected is Windscoop Bl

In [41]:
import pickle
# corpus_embeddings = sentbert_model.encode(train_descriptions, convert_to_tensor=True)
# with open('corpus_sentbert_embeddings_all.pickle', 'wb') as f:
#     pickle.dump(corpus_embeddings, f)
corpus_embeddings = pickle.load(open('corpus_sentbert_embeddings_all.pickle', 'rb'))

# Set Up - Prompt

In [42]:
sys_prompt = "A chat."
prompt = "Tell me a joke."

prefix = "<|im_start|>"
suffix = "<|im_end|>\n"
sys_format = prefix + "system\n" + sys_prompt + suffix
user_format = prefix + "user\n" + prompt + suffix
assistant_format = prefix + "assistant\n"
input_text = sys_format + user_format + assistant_format

generation_config = GenerationConfig(
    max_length=256, temperature=1.1, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")


In [43]:
def generate_answer(system_prompt, prompt):
  sys_format = prefix + "system\n" + sys_prompt + suffix
  user_format = prefix + "user\n" + prompt + suffix
  assistant_format = prefix + "assistant\n"
  input_text = sys_format + user_format + assistant_format

  generation_config = GenerationConfig(
    max_new_tokens=100, temperature=0.0001, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")

  inputs = tokenizer(input_text, return_tensors="pt", return_attention_mask=True).to(device)
  outputs = model.generate(**inputs, generation_config=generation_config)

  text = tokenizer.batch_decode(outputs)[0]
  text = text.split("<|im_end|>")[2][len("<|im_start|> assistant"):].strip("\n")
  print(text)

  return text

In [44]:
# Summarize And Explain
def add_taxonomy_description(description, description_to_taxonomy):
  # add taxonomy description if exists and see if that helps
  if description in description_to_taxonomy and type(description_to_taxonomy[description]) == str:
    term = description_to_taxonomy[description]
    system_prompt = "You are a scientist. User will give you a geology term. You must generate a short description of the term."
    instruction = f"""You are a scientist. Your task is to give a brief one sentence description of material the geology term it consists.
    ###
    <<<
    Term:{term}
    >>>
    """
    return generate_answer(system_prompt, instruction).strip("\n")
  else:
    return None

def add_collectionMethod_description(description, description_to_cm):
  # add collectionMethod description if exists and see if that helps
  if description in  description_to_cm and type( description_to_cm[description]) == str:
    term = description_to_cm[description]
    system_prompt = "You are a scientist. User will give you a term that indicates how it collected a sample from nature. You must generate a short description of the term."
    instruction = f"""You are a scientist. Your task is to give a brief one sentence description of the given collection method of the sample.
    ###
    <<<
    Term:{term}
    >>>
    """
    return generate_answer(system_prompt, instruction).strip("\n")
  else:
    return None
def add_description_description(description, description_to_desc):
  # add collectionMethod description if exists and see if that helps
  if description in  description_to_desc and type( description_to_desc[description]) == str:
    term = description_to_desc[description]
    system_prompt = "You are a scientist. User will give you a term that indicates a description of the sample. You must generate a short explanation of that description."
    instruction = f"""You are a scientist. Your task is to give a one sentence explanation of the description of the sample.
    ###
    <<<
    Description:{term}
    >>>
    """
  else:
    return None


def generate_summary(sample_description):
  system_prompt = "You are a scientist. User will give you a description of a material sample it sampled from the nature. You must generate a summarized description of the sample."
  prompt = f"""You are a scientist. Your task is to give a brief one sentence summary of the given description, focusing on the parts that is helpful in determining the type of material that constitutes it.
  Include important fields in the summary such as the taxonomy informal classification, collection method, and values that determine the material type.
  ###
  <<<
  Description: {sample_description}
  >>>
  """

  return generate_answer(system_prompt, prompt)

def generate_summary_and_explanation(sample_description):
  summary = generate_summary(sample_description)

  taxonomy_rich_description = add_taxonomy_description(sample_description, desc_to_tax_map)
  if taxonomy_rich_description:
    summary += taxonomy_rich_description.strip("\n")
  collectionMethodDesc = add_collectionMethod_description(sample_description, desc_to_cm_map)
  if collectionMethodDesc:
    summary += collectionMethodDesc.strip("\n")
  description_rich_description = add_description_description(sample_description, desc_to_desc_map)
  if description_rich_description:
    summary += description_rich_description.strip("\n")
  print("Enriched summary : ", summary)
  return summary

In [45]:
def generate_prediction(material_types, examples, sample_description):
  system_prompt = "You are a scientist. User will give you a task. You must generate an answer to the task."
  instruction = f"""
  You are a scientist. Your task is to analyze the description of a material sample and determine the kind of material that constitutes it after <<<>>> into one of the predefined material types: \n
  {material_types} \n
  You will only respond with the material type. Do not include the word "Material type". Do not provide explanations or notes.
  ###\n
  Here are some examples:
  {examples}
  ### \n
  <<<
  Description: {sample_description}
  Material type:
  >>>
  """
  return generate_answer(system_prompt, instruction).strip("\n")

In [46]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.metrics import jaccard_distance

def jaccard_similarity(set1, set2):
    """
    Calculate Jaccard similarity between two sets.
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def tokenize_text(text):
    """
    Tokenize input text.
    """
    return set(word_tokenize(text.lower()))

def jaccard_highest_score(sample_text, set_of_texts):
    """
    Find the text with the highest Jaccard similarity score compared to the sample text.
    """
    sample_tokens = tokenize_text(sample_text)
    highest_score = 0
    most_similar_text = None
    print(set_of_texts)
    for text in set_of_texts:
        text_tokens = tokenize_text(text)
        similarity_score = jaccard_similarity(sample_tokens, text_tokens)

        if similarity_score > highest_score:
            highest_score = similarity_score
            most_similar_text = text

    return highest_score, most_similar_text # most similar material type

def extract_prediction(candidate_material_types, response):
  # do greedy text match
  result = []
  response = response.lower()
  if response.startswith("The material type is: "):
    response = response[len("The material type is: ")]
  for candidate_material_type in candidate_material_types:
    if candidate_material_type in response:
      result.append(candidate_material_type)
    # convert multi label type labels
    elif candidate_material_type == "rock or sediment":
      if "rock" in response.lower() or "sediment" in response.lower():
        result.append(candidate_material_type)
    elif candidate_material_type == "mixed soil sediment or rock" and "soil" in response.lower():
      result.append(candidate_material_type)
  if len(result) == 0:
    # use jaccard score
    higest_score, most_similar_material_types = jaccard_highest_score(response, candidate_material_types)
    #print(f"Jaccard score : {higest_score} : {most_similar_material_types}")
    if higest_score >= 0.5:
      result.append(most_similar_material_types)

  return result

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [47]:
# function to generate few shot
import pickle
INPUT_LIMIT = 2048
def count_words(input_string):
    words = input_string.split(" ")
    return len(words)
def generate_few_shot_examples_sentbert(test_description, k):
  # use sentence transformers to find k closest few shot examples
  query_embedding = sentbert_model.encode(test_description, convert_to_tensor=True)

  search_hits = util.semantic_search(query_embedding, corpus_embeddings)
  search_hits = search_hits[0]  # Get the hits for the first query

  examples = ""
  for i in range(k-1, -1, -1): # from least similar
    sample_id = search_hits[i]['corpus_id']
    sample_description = train_descriptions[sample_id]
    sample_label = ",".join(final_desc_to_label[sample_description].split("/"))

    examples += "\nDescription: " + sample_description + "\nMaterial type: " + sample_label

    if count_words(examples) >= INPUT_LIMIT:
      break

  return examples


In [48]:
def few_shot_prediction(test_df, num_shot, summarize_and_explain = False, OUTPUT_FILE="output.json"):
  """
  num_shot : number of examples to use in the prompt
  summarize_and_explain : whether to summarize and explain the input text and use that instead of original sample_description
  """
  test_prediction = []
  for idx, row in test_df.iterrows():
    sample_description = row['concatenated_text_B'].split("</s>")[0][len("<s>"):]
    input = sample_description

    # SummarizeExplain
    if summarize_and_explain:
      summary = generate_summary_and_explanation(sample_description)
      input = summary

    examples = generate_few_shot_examples_sentbert(sample_description, num_shot)
    output = generate_prediction(joined_leaf_material_types,examples,input)

    prediction = extract_prediction(leaf_material_types, output)

    final_prediction = []
    if len(prediction) > 0:
      for pred in prediction:
        final_prediction.extend(leaf_to_entire_path_mapping[pred]) # get entire parents as well and add it to the prediction

    # TODO
    else:
      # level-up traversal recursively
      curr_labels = leaf_material_types
      parent_labels = get_parent_labels(curr_labels)

      while len(prediction) == 0 and len(parent_labels)>0:
        curr_labels = parent_labels
        joined_curr_labels = "\n".join(curr_labels)
        output = generate_prediction(joined_curr_labels,examples,input)

        prediction = extract_prediction(curr_labels, output)
        print("Extracted labels: ", prediction)
        final_prediction = []
        for pred in prediction:
          final_prediction.append(pred)
          final_prediction.extend(label_to_parent[pred])

        # recurse up one level
        parent_labels = get_parent_labels(curr_labels)


    final_prediction = list(set(final_prediction))
    final_prediction = [x for x in final_prediction if x!= None and x!='material']
    print("Prediction : ", final_prediction)

    test_prediction.append(final_prediction)
    print("Gold: ",row["label_list"])
    print(f"{idx}-th prediction done\n\n")
    if idx % 100 == 0:
      with open(OUTPUT_FILE, 'w') as file:
        json.dump(test_prediction, file)

  with open(OUTPUT_FILE, 'w') as file:
    json.dump(test_prediction, file)

  return test_prediction

test_prediction = few_shot_prediction(test_df, 5, False, OUTPUT_FILE)

 sediment,igneous rock,natural solid material,foiditoid,fine grained igneous rock,rock
Prediction :  ['fine grained igneous rock', 'foiditoid', 'igneous rock', 'rock or sediment', 'rock', 'natural solid material']
Gold:  sediment/igneous rock/natural solid material/foiditoid/fine grained igneous rock/rock
0-th prediction done


 diamictite,clastic sedimentary rock,sediment,natural solid material,sedimentary rock,rock
Prediction :  ['sedimentary rock', 'rock or sediment', 'diamictite', 'rock', 'clastic sedimentary rock', 'natural solid material']
Gold:  diamictite/clastic sedimentary rock/sediment/natural solid material/sedimentary rock/rock
1-th prediction done


 sediment,igneous rock,natural solid material,ultramafic igneous rock,peridotite,rock
Prediction :  ['ultramafic igneous rock', 'peridotite', 'igneous rock', 'rock or sediment', 'rock', 'natural solid material']
Gold:  sediment/igneous rock/natural solid material/ultramafic igneous rock/peridotite/rock
2-th prediction done


 

KeyboardInterrupt: 

# Evaluate

In [ ]:
multi_to_label = {
    "rock or sediment": ["rock", "sediment"],
    "mixed soil sediment rock" : ["soil", "sediment", "rock"]
}

final_predicted_labels = test_prediction
for idx, labels in enumerate(test_prediction):
  for label in labels:
    if label in multi_to_label:
      labels.remove(label)
      labels.extend(multi_to_label[label])
      labels = list(set(labels))
  final_predicted_labels[idx] = labels # update

# assert
for idx, labels in enumerate(test_prediction):
  for label in labels:
    if label in multi_to_label:
      print("invalid")
      break
final_predicted_labels[-1]

In [ ]:
label_file="total_labels.txt" # file that stores all labels of iSamples taxonomy
gold_label_names = open(label_file).read().splitlines()

In [ ]:
true_labels = [x.split("/") for x in test_df['label_list'].tolist()]
true_labels

In [ ]:
## Multi label evaluation
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit([gold_label_names])

true_labels_bin = mlb.transform(true_labels)
predicted_labels_bin = mlb.transform(final_predicted_labels)

print(classification_report(true_labels_bin, predicted_labels_bin, target_names=mlb.classes_))
report = classification_report(true_labels_bin, predicted_labels_bin, target_names=mlb.classes_, output_dict=True)

# Print the classification report
print(report)